In [1]:
from astropy import units as u
from astropy.constants import G, c
import numpy as np
from numpy import pi
import pandas as pd
import spacehub_calc as sc

In [2]:
M = 1e8 * u.Msun
Rg = G*M / c / c

#sample orbit
a = 1e3 * Rg
e = 0.1
omega = pi/4

RCross1 = a * (1-e**2) / (1 + e*np.cos(-omega))
RCross1 = a * (1-e**2) / (1 + e*np.cos(pi-omega))


In [3]:
m = 10 * u.Msun
r_part = (2*G*m/c**2).to(u.m)
i_orb = pi/3
Omega_node = 0.0

T_unit = u.yr / (2*pi)
disk = pd.read_csv('SpaceHub/src/interaction/disk_tab/SG_01Edd.csv')
R_grid     = (disk['R'].values * u.AU).to(u.m).value
rho_grid   = (disk['rho'].values * u.Msun / u.AU**3).to(u.kg/u.m**3).value
cs_grid    = (disk['cs'].values  * u.AU / T_unit).to(u.m/u.s).value
gradP_grid = disk['grad_P'].values

def rho_of(R):   return np.interp(R.to(u.m).value, R_grid, rho_grid) * u.kg/u.m**3
def cs_of(R):    return np.interp(R.to(u.m).value, R_grid, cs_grid)  * u.m/u.s
def gradP_of(R): return np.interp(R.to(u.m).value, R_grid, gradP_grid)

def disk_v_vec(rvec):
    Rcyl = np.sqrt(rvec[0]**2 + rvec[1]**2)
    v_k  = np.sqrt(G*M/Rcyl)
    n    = gradP_of(Rcyl)
    cs   = cs_of(Rcyl)
    v_d  = v_k * np.sqrt(1 - n*(cs/v_k)**2)
    return u.Quantity([-v_d*rvec[1]/Rcyl, v_d*rvec[0]/Rcyl, 0*v_d]).to(u.m/u.s)

def state_at(f):
    p    = a*(1 - e**2)
    rmag = p/(1 + e*np.cos(f))
    r_pf = u.Quantity([rmag*np.cos(f), rmag*np.sin(f), 0*rmag])
    v0   = np.sqrt(G*M/p)
    v_pf = u.Quantity([-v0*np.sin(f), v0*(e + np.cos(f)), 0*v0])
    cO, sO = np.cos(Omega_node), np.sin(Omega_node)
    ci, si = np.cos(i_orb),      np.sin(i_orb)
    cw, sw = np.cos(omega),      np.sin(omega)
    Rmat = np.array([[cO*cw - sO*sw*ci, -cO*sw - sO*cw*ci,  sO*si],
                     [sO*cw + cO*sw*ci, -sO*sw + cO*cw*ci, -cO*si],
                     [sw*si,             cw*si,             ci   ]])
    return (Rmat @ r_pf).to(u.m), (Rmat @ v_pf).to(u.m/u.s)

logL = 3.0
eps_M = np.exp(-2*logL/3)
def I_sup(M_):  return (0.5*np.log(1 - 1/M_**2) + logL)/M_**2
def I_sub(M_):  return (0.5*np.log((1 + M_)/(1 - M_)) - M_)/M_**2
def dI_sup(M_): return (-2*logL + 1/(M_**2 - 1) - np.log(1 - 1/M_**2))/M_**3
def dI_sub(M_): return (M_**3 + (1 - M_**2)*np.log((1+M_)/(1-M_)) - 2*M_)/(M_**3*(M_**2 - 1))
x1, x2 = 1 - eps_M, 1 + eps_M
y1, y2 = I_sub(x1),  I_sup(x2)
k1, k2 = dI_sub(x1), dI_sup(x2)
A_h = k1*(x2 - x1) - (y2 - y1)
B_h = -k2*(x2 - x1) + (y2 - y1)
def alpha(M_):
    if M_ <= 0.1: return M_/3.0
    if M_ < x1:   return I_sub(M_)
    if M_ > x2:   return I_sup(M_)
    t = (M_ - x1)/(x2 - x1)
    return (1 - t)*y1 + t*y2 + (1 - t)*t*(t*B_h + (1 - t)*A_h)
def beta(M_):    return 1.0/(1.0 + M_**2)

def F_drag(f):
    rvec, vpart = state_at(f)
    vrel = vpart - disk_v_vec(rvec)
    vmag = np.sqrt(np.sum(vrel**2))
    Rcyl = np.sqrt(rvec[0]**2 + rvec[1]**2)
    cs   = cs_of(Rcyl)
    rho  = rho_of(Rcyl)
    Mach = float((vmag/cs).to(u.dimensionless_unscaled))
    R_eff = max(r_part, (G*m/(vmag**2 + cs**2)).to(u.m))
    F_HL  = 4*pi*G**2*m**2*rho/cs**2
    F_ae  = pi*R_eff**2*rho*vmag**2
    Fmag  = F_HL*(alpha(Mach) + beta(Mach)) + F_ae
    return (-Fmag * vrel/vmag).to(u.N)

F1 = F_drag(-omega)
F2 = F_drag(pi - omega)
F1, F2

(<Quantity [ 1.15551168e+23,  8.16060976e+23, -1.51527725e+24] N>,
 <Quantity [ 1.39047937e+23, -9.81918581e+23,  1.58256343e+24] N>)